# Stage 3: EDA and data quality

This notebook checks the structure, quality, and limitations of the Bank Marketing data before modelling.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_bank

sns.set_theme(style="whitegrid")
df = load_bank()
df.head()

## 1. Structure and data types

In [ ]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nNumeric summary:")
display(df.describe().T)

## Observation and decision

The data contains client information collected before a call, plus the target `y`. The numeric and categorical columns will need different preprocessing later. The `duration` column is retained for quality inspection but must not be used as a model feature because it is only known after the call.

## 2. Variable descriptions

In [ ]:
variable_descriptions = [
    ("age", "numeric", "Client age in years"),
    ("job", "categorical", "Type of job"),
    ("marital", "categorical", "Marital status"),
    ("education", "categorical", "Highest recorded education level"),
    ("default", "categorical", "Whether the client has credit in default"),
    ("balance", "numeric", "Average yearly balance in euros"),
    ("housing", "categorical", "Whether the client has a housing loan"),
    ("loan", "categorical", "Whether the client has a personal loan"),
    ("contact", "categorical", "Contact communication type"),
    ("day", "numeric", "Day of the month when the client was contacted"),
    ("month", "categorical", "Month when the client was contacted"),
    ("duration", "numeric", "Call duration in seconds; known only after the call"),
    ("campaign", "numeric", "Number of contacts during this campaign"),
    ("pdays", "numeric", "Days since the client was previously contacted"),
    ("previous", "numeric", "Number of contacts before this campaign"),
    ("poutcome", "categorical", "Outcome of the previous marketing campaign"),
    ("y", "target", "Whether the client subscribed to a term deposit"),
]
variable_table = pd.DataFrame(variable_descriptions, columns=["Variable", "Type", "Plain-language description"])
display(variable_table)

## Observation and decision

The table documents every variable in plain language. The target is separate from the 15 raw pre-call features, and `duration` is excluded from modelling to prevent target leakage.

## 3. Target and class imbalance

In [ ]:
target_counts = df["y"].value_counts().rename_axis("subscription").reset_index(name="count")
target_counts["percent"] = target_counts["count"] / len(df) * 100
display(target_counts)

sns.countplot(data=df, x="y", order=["no", "yes"])
plt.title("Term-deposit subscription counts")
plt.xlabel("Subscribed?")
plt.ylabel("Number of clients")
plt.show()

## Observation and decision

The `yes` class is much smaller than the `no` class, so this is an imbalanced classification problem. Model comparison must include precision, recall, F1, PR-AUC, and ROC-AUC rather than accuracy alone. The split will be stratified so both sets keep a similar class balance.

## 4. Missing values and special values

In [ ]:
missing_values = df.isna().sum().sort_values(ascending=False)
missing_table = missing_values[missing_values > 0].rename("missing_count").to_frame()
if missing_table.empty:
    print("No missing values remain after load_bank() cleaning.")
else:
    display(missing_table)

categorical_columns = df.select_dtypes(include="object").columns
unknown_counts = (df[categorical_columns] == "unknown").sum().sort_values(ascending=False)
display(unknown_counts[unknown_counts > 0].rename("unknown_count").to_frame())

print(f"pdays = -1 count: {(df['pdays'] == -1).sum()}")
print("Meaning: 'unknown' means a categorical value was unavailable; pdays = -1 means the client was not previously contacted.")

## Observation and decision

`unknown` is treated as an informative category for missing categorical information, rather than silently dropping those rows. A `pdays` value of `-1` means there was no previous contact, so it is a meaningful sentinel value and should not be treated as an ordinary positive number.

## 5. Duplicate rows

In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")
print(f"Duplicate percentage: {duplicate_count / len(df) * 100:.2f}%")

## Observation and decision

Exact duplicate rows should not provide additional evidence about a client. We will record their count and decide whether to remove them before modelling after checking whether the duplicates are genuine repeated records or data-entry duplicates.

## 6. Outliers and IQR counts

In [ ]:
outlier_columns = ["balance", "campaign", "previous", "pdays"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for axis, column in zip(axes.ravel(), outlier_columns):
    sns.boxplot(data=df, y=column, ax=axis)
    axis.set_title(f"{column} boxplot")
plt.tight_layout()
plt.show()

iqr_rows = []
for column in outlier_columns:
    first_quartile = df[column].quantile(0.25)
    third_quartile = df[column].quantile(0.75)
    iqr = third_quartile - first_quartile
    lower_bound = first_quartile - 1.5 * iqr
    upper_bound = third_quartile + 1.5 * iqr
    count = ((df[column] < lower_bound) | (df[column] > upper_bound)).sum()
    iqr_rows.append((column, lower_bound, upper_bound, count))

iqr_table = pd.DataFrame(iqr_rows, columns=["Variable", "Lower IQR bound", "Upper IQR bound", "IQR outlier count"])
display(iqr_table)

## Observation and decision

The boxplots and IQR counts identify unusually large or small values, but an IQR flag is not automatically an error. For example, high balances or many campaign contacts may be real clients. We will inspect these values and use preprocessing that is robust to skew rather than deleting observations only because they are flagged.